# 02 — LiteLLM basic (gateway, not Auto Router)

Evaluate **LiteLLM as a multi-provider routing framework**: one API, aliases, retries, fallbacks, multiple local Ollama models + OpenAI.

This is **not** Auto Router v2 (that is notebook 06). Do **not** use shuffle / latency / usage load-balancing as the intent classifier — those pick among redundant clones, not among differently capable models.

Pull a second local tag if you want `local-alt` to be a real other model:

```
ollama pull llama3   # or whatever OLLAMA_MODEL / OLLAMA_MODEL_2 are
```

Kernel: Python 3.10+.


In [1]:
# %pip install 'litellm>=1.50' python-dotenv -q


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd()
poc_dir = None
root = None
for p in [cwd, *cwd.parents]:
    if (p / "eval_queries.py").exists():
        poc_dir = p
        break
    if (p / "poc" / "eval_queries.py").exists():
        poc_dir = p / "poc"
        break
if poc_dir is None:
    raise FileNotFoundError("eval_queries.py not found — run from route-chatbot/ or route-chatbot/poc/")
sys.path.insert(0, str(poc_dir))

for p in [cwd, *cwd.parents]:
    if (p / ".env").exists() and (p / "main.py").exists():
        root = p
        load_dotenv(p / ".env")
        break
else:
    load_dotenv()

from eval_queries import EVAL_QUERIES

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3")
OLLAMA_MODEL_2 = os.getenv("OLLAMA_MODEL_2", "llama3")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-3.5-turbo")
TYPESAFE_API_KEY = os.getenv("TYPESAFE_API_KEY", "")

print("poc_dir", poc_dir)
print("eval queries", len(EVAL_QUERIES))
print("ollama", OLLAMA_BASE_URL, OLLAMA_MODEL, "| alt", OLLAMA_MODEL_2)
print("openai model", OPENAI_MODEL, "| key set", bool(OPENAI_API_KEY))
print("typesafe key set", bool(TYPESAFE_API_KEY.strip()))
print("typesafe key set", bool(TYPESAFE_API_KEY.strip()))


poc_dir /Users/tushar/PravarAI/route-chatbot/poc
eval queries 17
ollama http://localhost:11434 llama3.1:8b | alt qwen3.5:9b
openai model gpt-3.5-turbo | key set True
typesafe key set False
typesafe key set False


## Router: aliases, retries, fallbacks


In [3]:
from litellm import Router

model_list = [
    {
        "model_name": "local-fast",
        "litellm_params": {
            "model": f"ollama/{OLLAMA_MODEL}",
            "api_base": OLLAMA_BASE_URL,
        },
    },
    {
        "model_name": "local-alt",
        "litellm_params": {
            "model": f"ollama/{OLLAMA_MODEL_2}",
            "api_base": OLLAMA_BASE_URL,
        },
    },
    {
        "model_name": "openai",
        "litellm_params": {
            "model": OPENAI_MODEL,
            "api_key": OPENAI_API_KEY,
        },
    },
]

# If local-fast is down, try the other Ollama tag, then OpenAI.
router = Router(
    model_list=model_list,
    num_retries=2,
    fallbacks=[{"local-fast": ["local-alt", "openai"]}],
)

print("deployments:", [m["model_name"] for m in model_list])
print("fallbacks: local-fast -> local-alt -> openai")


deployments: ['local-fast', 'local-alt', 'openai']
fallbacks: local-fast -> local-alt -> openai


## Intent pick is still ours; LiteLLM does the call

A tiny mapper chooses an **alias**. LiteLLM owns provider URLs, retries, and fallbacks. That split is the point of this notebook: LiteLLM is the gateway, not the classifier.


In [4]:
import re

GREETING_PATTERN = re.compile(
    r"^\s*(hi|hello|hey|good morning|good afternoon|good evening|thanks|thank you|bye|how are you)\b",
    re.IGNORECASE,
)
COMPLEX_KEYWORDS = (
    "code", "function", "debug", "algorithm", "write a", "explain why",
    "compare", "analyze", "design", "poem", "story", "essay", "strategy",
)


def pick_alias(message: str) -> str:
    if GREETING_PATTERN.search(message):
        return "local-fast"
    lower = message.lower()
    if any(keyword in lower for keyword in COMPLEX_KEYWORDS):
        return "openai"
    return "local-fast"


def decide_route(message: str) -> str:
    alias = pick_alias(message)
    return "openai" if alias == "openai" else "ollama"


## Eval (routing only — no LiteLLM completion)


In [5]:
rows = []
for item in EVAL_QUERIES:
    t0 = time.perf_counter()
    err = None
    predicted = None
    extra = None
    try:
        result = decide_route(item["message"])
        if isinstance(result, tuple):
            predicted = result[0]
            extra = result[1] if len(result) > 1 else None
        else:
            predicted = result
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
    ms = (time.perf_counter() - t0) * 1000
    rows.append({
        "message": item["message"],
        "expected": item["expected"],
        "predicted": predicted,
        "match": predicted == item["expected"],
        "latency_ms": round(ms, 1),
        "error": err,
        "extra": extra,
    })

n = len(rows)
ok = sum(1 for r in rows if r["match"])
errs = sum(1 for r in rows if r["error"])
mean_ms = sum(r["latency_ms"] for r in rows) / n if n else 0
print(f"accuracy {ok}/{n} ({100 * ok / n:.0f}%)  mean latency {mean_ms:.1f} ms  errors {errs}")
print()
for r in rows:
    flag = "OK  " if r["match"] else "MISS"
    extra = f"  {r['extra']}" if r["extra"] else ""
    err = f"  ERR {r['error']}" if r["error"] else ""
    print(f"  [{flag}] {r['latency_ms']:7.1f} ms  exp={r['expected']:7} pred={r['predicted']}  {r['message'][:70]}{extra}{err}")


accuracy 16/17 (94%)  mean latency 0.0 ms  errors 0

  [OK  ]     0.0 ms  exp=ollama  pred=ollama  hi there
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  hello
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  hey
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  good morning
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  thanks
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  how are you
  [OK  ]     0.0 ms  exp=openai  pred=openai  write a function to reverse a linked list
  [OK  ]     0.0 ms  exp=openai  pred=openai  compare merge sort and quick sort
  [OK  ]     0.0 ms  exp=openai  pred=openai  debug this python code
  [OK  ]     0.0 ms  exp=openai  pred=openai  analyze the time complexity of this algorithm
  [OK  ]     0.0 ms  exp=openai  pred=openai  write a poem about the ocean
  [OK  ]     0.0 ms  exp=openai  pred=openai  design a strategy for caching
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  tell me something interesting
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  what did you do to

## Notes (fill during the experiment)

LiteLLM advantages to look at (vs hard-coded clients in `main.py`):

- One `router.completion(model=alias, ...)` for Ollama and OpenAI
- Multiple Ollama models as first-class deployments
- Retries + fallbacks if a local model is down
- Swap a provider by editing `model_list`, not client classes

Not an advantage here: load-balancing strategies (`simple-shuffle`, latency, usage-v2) — wrong tool for intent.


In [6]:
GENERATE = True  # flip during the experiment

if GENERATE:
    for item in EVAL_QUERIES[:]:
        alias = pick_alias(item["message"])
        print("---", item["message"], "->", alias)
        resp = router.completion(
            model=alias,
            messages=[{"role": "user", "content": item["message"]}],
            max_tokens=64,
        )
        print("served by", resp.model)
        print((resp.choices[0].message.content or "")[:400])
        print()
else:
    print("GENERATE is False — routing only. Flip it to call router.completion.")


--- hi there -> local-fast
served by ollama/llama3.1:8b
hi how's it going?

--- hello -> local-fast
served by ollama/llama3.1:8b
hello! how can i assist you today?

--- hey -> local-fast
served by ollama/llama3.1:8b
hey! how's it going?

--- good morning -> local-fast
served by ollama/llama3.1:8b
Good morning! How are you today?

--- thanks -> local-fast
served by ollama/llama3.1:8b
You're welcome! Is there anything else I can help you with?

--- how are you -> local-fast
served by ollama/llama3.1:8b
I'm just a language model, I don't have emotions or feelings like humans do, but I'm functioning properly and ready to help with any questions or tasks you may have! How about you? How's your day going?

--- write a function to reverse a linked list -> openai
served by gpt-3.5-turbo-0125
Here is a Python function to reverse a linked list:

```python
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None

class LinkedList:
    def __init__(self):
    